In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

## CÓDIGO VERSÃO 1.1 - Desenvolvido por Emanuel Gomes Soares - Engenheiro Ambiental

# Lista de estados válidos conforme estrutura da FBDS
VALID_STATES = {
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT",
    "PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"
}

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

def download_file(url, folder):
    """Baixa um arquivo e salva na pasta especificada."""
    try:
        os.makedirs(folder, exist_ok=True)
        filename = os.path.basename(urlparse(url).path)
        local_path = os.path.join(folder, filename)

        if os.path.exists(local_path):
            print(f"[SKIP] Já existe: {local_path}")
            return

        print(f"[DOWN] {url}")
        resp = session.get(url, timeout=20, stream=True)
        resp.raise_for_status()

        with open(local_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"[OK] Salvo: {local_path}")

    except Exception as e:
        print(f"[ERRO] Falha ao baixar {url}: {e}")

def is_valid_relative_link(href):
    """Verifica se o href é um link relativo válido."""
    if not href or href.startswith(("http://", "https://", "//", "?")):
        return False
    if ":" in href.split("/")[0]:
        return False
    return True

def scrape_directory_parallel(start_url, base_folder):
    """Varre diretórios e baixa arquivos usando processamento paralelo."""
    queue = [(start_url, base_folder)]
    download_tasks = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        while queue:
            url, folder = queue.pop(0)
            print(f"[DIR] Acessando: {url}")

            try:
                resp = session.get(url, timeout=20)
                resp.raise_for_status()
            except Exception as e:
                print(f"[ERRO] Não acessou {url}: {e}")
                continue

            soup = BeautifulSoup(resp.text, "html.parser")
            links = soup.find_all("a", href=True)

            for link in links:
                href = link["href"]

                if not is_valid_relative_link(href):
                    continue

                full_url = urljoin(url, href)

                if href.endswith("/"):
                    new_folder = os.path.join(folder, href.rstrip("/"))
                    queue.append((full_url, new_folder))
                else:
                    download_tasks.append(
                        executor.submit(download_file, full_url, folder)
                    )

        for task in as_completed(download_tasks):
            pass

if __name__ == "__main__":
    print("=== Downloader FBDS - Versão 1.1 ===")
    print("Desenvolvido por Emanuel Gomes Soares - Engenheiro Ambiental\n")

    print("Estados disponíveis:")
    print(", ".join(sorted(VALID_STATES)))
    print()

    state = input("Digite a sigla do estado (ex: PB): ").upper().strip()

    if state not in VALID_STATES:
        print(f"[ERRO] Sigla '{state}' inválida! Encerrando o programa.")
        exit()

    BASE_URL = f"https://geo.fbds.org.br/{state}/"
    DOWNLOAD_DIR = f"{state}_download"

    print(f"\nEstado selecionado: {state}")
    print(f"URL base: {BASE_URL}")
    print(f"Pasta de destino: {DOWNLOAD_DIR}\n")

    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    scrape_directory_parallel(BASE_URL, DOWNLOAD_DIR)

    print("\n Download concluído!")


=== Downloader FBDS - Versão 1.1 ===
Desenvolvido por Emanuel Gomes Soares - Engenheiro Ambiental

Estados disponíveis:
AC, AL, AM, AP, BA, CE, DF, ES, GO, MA, MG, MS, MT, PA, PB, PE, PI, PR, RJ, RN, RO, RR, RS, SC, SE, SP, TO



Digite a sigla do estado (ex: PB):  AC



Estado selecionado: AC
URL base: https://geo.fbds.org.br/AC/
Pasta de destino: AC_download

[DIR] Acessando: https://geo.fbds.org.br/AC/
[DIR] Acessando: https://geo.fbds.org.br/AC/ACRELANDIA/[SKIP] Já existe: AC_download\

[DIR] Acessando: https://geo.fbds.org.br/AC/ASSIS_BRASIL/
[SKIP] Já existe: /AC/ACRELANDIA\
[DIR] Acessando: https://geo.fbds.org.br/AC/BRASILEIA/
[SKIP] Já existe: /AC/ASSIS_BRASIL\
[DIR] Acessando: https://geo.fbds.org.br/AC/BUJARI/
[SKIP] Já existe: /AC/BRASILEIA\
[DIR] Acessando: https://geo.fbds.org.br/AC/CAPIXABA/
[SKIP] Já existe: /AC/BUJARI\
[DIR] Acessando: https://geo.fbds.org.br/AC/CRUZEIRO_DO_SUL/
[SKIP] Já existe: /AC/CAPIXABA\
[DIR] Acessando: https://geo.fbds.org.br/AC/EPITACIOLANDIA/
[SKIP] Já existe: /AC/CRUZEIRO_DO_SUL\
[DIR] Acessando: https://geo.fbds.org.br/AC/FEIJO/
[SKIP] Já existe: /AC/EPITACIOLANDIA\
[DIR] Acessando: https://geo.fbds.org.br/AC/JORDAO/
[SKIP] Já existe: /AC/FEIJO\
[DIR] Acessando: https://geo.fbds.org.br/AC/MANCIO_LIMA/
[SKI